In [ ]:
import re
import pandas as pd
from collections import Counter
from pathlib import Path

# Đường dẫn file Silver
SILVER_FILE = Path("D:/topcv-data-engineer/data/silver/jobs_silver.parquet")

# Danh sách stopwords (các từ không mang ý nghĩa kỹ năng)
STOPWORDS = {
    'experience', 'requirement', 'description', 'job', 'work', 'team', 'skill',
    'ability', 'knowledge', 'proficiency', 'familiar', 'understanding',
    'strong', 'good', 'excellent', 'basic', 'advanced', 'intermediate',
    'có', 'kinh', 'nghiệm', 'yêu', 'cầu', 'ứng', 'viên', 'thành', 'thạo',
    'hiểu', 'biết', 'tốt', 'nghiệp', 'vụ', 'chuyên', 'môn', 'vị', 'trí',
    'công', 'việc', 'và', 'hoặc', 'với', 'cho', 'của', 'một', 'các', 'những',
    'từ', 'đến', 'trong', 'ngoài', 'trên', 'dưới', 'theo', 'qua', 'bằng',
    'được', 'có', 'không', 'cũng', 'đã', 'sẽ', 'đang', 'là', 'ở'
}

def load_data(file_path):
    """Đọc dữ liệu từ Parquet hoặc CSV."""
    if file_path.suffix == '.parquet':
        return pd.read_parquet(file_path)
    elif file_path.suffix == '.csv':
        return pd.read_csv(file_path)
    else:
        raise ValueError("Unsupported file format. Use .parquet or .csv")

def combine_text(df):
    """Ghép description và requirements thành một chuỗi."""
    df_desc = df['description'].fillna('')
    df_req = df['requirements'].fillna('')
    return (df_desc + ' ' + df_req).tolist()

def tokenize_text(text):
    """Tách từ bằng regex, lọc stopwords và token quá ngắn."""
    # Tìm các từ/cụm có chữ cái hoặc số, cho phép dấu gạch nối và gạch dưới
    token_pat = re.compile(r'[A-Za-z0-9]+(?:[-_][A-Za-z0-9]+)*')
    tokens = token_pat.findall(text)
    tokens = [t.lower() for t in tokens if len(t) > 2 and t not in STOPWORDS]
    return tokens

def extract_ngrams(texts, n_range=(1, 3), min_freq=3, max_features=500):
    """Trích xuất n-grams (1-3 từ) từ danh sách văn bản."""
    all_tokens = [tokenize_text(t) for t in texts]
    
    freq_counter = Counter()
    for tokens in all_tokens:
        for n in range(n_range[0], n_range[1] + 1):
            for i in range(len(tokens) - n + 1):
                gram = ' '.join(tokens[i:i+n])
                freq_counter[gram] += 1
    
    # Lọc theo tần suất và giới hạn số lượng
    sorted_items = sorted(freq_counter.items(), key=lambda x: x[1], reverse=True)
    filtered = [(k, v) for k, v in sorted_items if v >= min_freq]
    return dict(filtered[:max_features])

def filter_candidates(freq_dict):
    """Lọc các ứng viên kỹ năng (loại bỏ số, từ quá ngắn)."""
    candidates = {}
    for phrase, count in freq_dict.items():
        if re.match(r'^\d+$', phrase):
            continue
        if len(phrase) <= 2:
            continue
        if ' ' not in phrase and len(phrase) < 4 and count < 5:
            continue
        candidates[phrase] = count
    return dict(sorted(candidates.items(), key=lambda x: x[1], reverse=True))

def main():
    print(f"Đang đọc dữ liệu từ {SILVER_FILE}")
    df = load_data(SILVER_FILE)
    print(f"Tổng số job: {len(df)}")

    texts = combine_text(df)
    print(f"Đã kết hợp description và requirements cho {len(texts)} job")

    print("Đang trích xuất n-grams (unigrams, bigrams, trigrams)...")
    freq_dict = extract_ngrams(texts, n_range=(1, 3), min_freq=3, max_features=500)
    print(f"Số lượng cụm từ duy nhất: {len(freq_dict)}")

    candidates = filter_candidates(freq_dict)
    print(f"Số lượng ứng viên kỹ năng tiềm năng: {len(candidates)}")
    print("\nTop 30 ứng viên kỹ năng theo tần suất:")
    for i, (phrase, count) in enumerate(list(candidates.items())[:30], 1):
        print(f"{i:2d}. {phrase}: {count} lần")

    output_file = Path("skill_candidates.txt")
    output_file.parent.mkdir(parents=True, exist_ok=True)
    with open(output_file, 'w', encoding='utf-8') as f:
        f.write("=== ỨNG VIÊN KỸ NĂNG TIỀM NĂNG ===\n")
        f.write(f"Tổng số ứng viên: {len(candidates)}\n\n")
        for phrase, count in candidates.items():
            f.write(f"{phrase}: {count}\n")
    print(f"\nĐã lưu danh sách ứng viên vào {output_file}")

if __name__ == "__main__":
    main()

Đang đọc dữ liệu từ D:\topcv-data-engineer\data\silver\jobs_silver.parquet
Tổng số job: 50
Đã kết hợp description và requirements cho 50 job
Đang trích xuất n-grams (unigrams, bigrams, trigrams)...
Số lượng cụm từ duy nhất: 500
Số lượng ứng viên kỹ năng tiềm năng: 499

Top 30 ứng viên kỹ năng theo tần suất:
 1. data: 538 lần
 2. nghi: 486 lần
 3. and: 294 lần
 4. thi: 186 lần
 5. tri: 164 lần
 6. quy: 153 lần
 7. ngh: 147 lần
 8. chuy: 139 lần
 9. tuy: 132 lần
10. thu: 121 lần
11. with: 97 lần
12. khai: 95 lần
13. quan: 95 lần
14. sql: 92 lần
15. etl: 88 lần
16. gia: 84 lần
17. tuy tuy: 84 lần
18. tin: 83 lần
19. the: 80 lần
20. aws: 78 lần
21. tri khai: 77 lần
22. gian: 72 lần
23. python: 69 lần
24. nhi: 68 lần
25. kinh: 61 lần
26. kinh nghi: 61 lần
27. pipeline: 60 lần
28. cloud: 59 lần
29. tham: 59 lần
30. tham gia: 58 lần

Đã lưu danh sách ứng viên vào skill_candidates.txt


: 

In [ ]:
import sys
import numpy as np
repo_root = r"D:\topcv-data-engineer"
sys.path.insert(0, repo_root)

from transform.src.gold.tables.fact_skills import _normalize_skills
sample = np.array(['api', 'mongodb', 'postgresql'])
result = _normalize_skills(sample)
print(result)  # Kỳ vọng: ['api', 'mongodb', 'postgresql']